# LangGraph Real Estate Agent Pipeline

This notebook builds a production-style LangGraph workflow for:
- scraping listing pages
- extracting and structuring data (without using source price)
- validating and enriching features
- running valuation through EstateMind services
- postprocessing and storage
- returning display-ready listings

Pipeline:
`scraper -> extraction -> structuring -> validation -> feature_engineering -> valuation -> postprocessing -> storage -> END`

In [1]:
# Optional: install extra dependencies if they are missing.
# Uncomment when needed in a fresh environment.
# %pip install langgraph langchain-core openai fastapi uvicorn

In [5]:
from __future__ import annotations

import asyncio
import json
import logging
import os
import re
import sqlite3
import sys
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, TypedDict

import requests
from bs4 import BeautifulSoup

from langgraph.graph import END, StateGraph

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project imports from EstateMind
from src.scripts.scraper import RequestManager, normalize_tunisian_data
from src.inference.valuation_service import ValuationService

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
)
LOGGER = logging.getLogger('langgraph_real_estate_agent')

In [13]:
class ListingAgentState(TypedDict, total=False):
    raw_html: str
    extracted_data: Dict[str, Any]
    structured_data: Dict[str, Any]
    validated_data: Dict[str, Any]
    enriched_features: Dict[str, Any]
    predicted_price: Dict[str, Any]
    final_listing: Dict[str, Any]
    logs: List[str]
    source_url: str
    warnings: List[str]


def _utc_now_iso() -> str:
    """Return the current UTC timestamp in ISO-8601 format."""
    return datetime.now(timezone.utc).isoformat()


def _append_log(state: ListingAgentState, message: str) -> None:
    """Append a timestamped message to the state log and notebook logger."""
    logs = state.setdefault('logs', [])
    stamped = f"{_utc_now_iso()} | {message}"
    logs.append(stamped)
    LOGGER.info(message)


def with_retries(
    func: Callable[..., Any],
    *args: Any,
    retries: int = 3,
    base_sleep: float = 1.0,
    **kwargs: Any,
) -> Any:
    """Call a function with exponential backoff and re-raise after the final attempt."""
    last_exc: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as exc:
            last_exc = exc
            if attempt == retries:
                break
            sleep_s = base_sleep * (2 ** (attempt - 1))
            LOGGER.warning('Retry %s/%s after error: %s', attempt, retries, exc)
            time.sleep(sleep_s)
    raise RuntimeError(f'Function {func.__name__} failed after {retries} retries') from last_exc

In [14]:
# Shared resources
REQUEST_MANAGER = RequestManager(max_retries=4, backoff_factor=1.0)
VALUATION_SERVICE: Optional[ValuationService]
try:
    VALUATION_SERVICE = ValuationService()
except Exception as exc:
    VALUATION_SERVICE = None
    LOGGER.warning('ValuationService could not initialize; notebook will use mock valuation fallback: %s', exc)

HISTORY_STORE: List[Dict[str, Any]] = []
DB_PATH = Path('artifacts/langgraph_listings.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)


def init_db(db_path: Path = DB_PATH) -> None:
    """Create the local SQLite table used to persist processed listings."""
    conn = sqlite3.connect(db_path)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS listings (
            id TEXT PRIMARY KEY,
            created_at TEXT NOT NULL,
            source_url TEXT NOT NULL,
            title TEXT,
            property_type TEXT,
            governorate TEXT,
            city TEXT,
            neighborhood TEXT,
            surface_m2 REAL,
            bedrooms INTEGER,
            bathrooms INTEGER,
            condition TEXT,
            predicted_price_tnd REAL,
            confidence REAL,
            payload_json TEXT NOT NULL
        )
        """
    )
    conn.commit()
    conn.close()


init_db()

In [ ]:
@dataclass
class ValuationPayload:
    """Structured input sent to the valuation service."""
    property_type: str
    transaction_type: str
    governorate: str
    city: str
    neighborhood: str
    size_m2: float
    bedrooms: int
    bathrooms: int
    condition: str
    has_pool: bool
    has_garden: bool
    has_parking: bool
    sea_view: bool
    elevator: bool
    description: str
    uploaded_images_count: int
    image_refs: List[str]

In [ ]:
def scraper_node(state: ListingAgentState) -> ListingAgentState:
    """Fetch listing HTML from the source URL or fall back to sample HTML."""
    _append_log(state, 'scraper_node: start')
    url = state.get('source_url', '').strip()
    if not url:
        raise ValueError('source_url is required in the state')

    try:
        response = with_retries(REQUEST_MANAGER.get, url, timeout=25, retries=3)
        state['raw_html'] = response.text
        _append_log(state, f'scraper_node: fetched {len(response.text)} chars from {url}')
    except Exception as exc:
        _append_log(state, f'scraper_node: failed request, using fallback HTML. error={exc}')
        state['raw_html'] = f"""
        <html><body>
          <h1>Bright apartment in La Marsa</h1>
          <p class='description'>Fully renovated 3-bedroom apartment with sea view and parking.</p>
          <span class='location'>Tunis, La Marsa</span>
          <span class='surface'>145 m2</span>
          <span class='rooms'>4</span>
          <span class='bathrooms'>2</span>
          <span class='condition'>Excellent</span>
          <span class='property_type'>Appartement</span>
        </body></html>
        """
    return state


def _extract_with_openai(html_text: str) -> Dict[str, Any]:
    """Use OpenAI to extract structured listing fields from raw HTML."""
    from openai import OpenAI

    prompt = (
        'Extract a Tunisian real estate listing into JSON. '
        'Do not include any price field. '
        'Return keys: title, description, property_type, governorate, city, neighborhood, '
        'surface_m2, bedrooms, bathrooms, condition, transaction_type, amenities, posted_at. '
        'If unknown, use null. HTML follows:\n' + html_text[:12000]
    )
    client = OpenAI()
    completion = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        messages=[
            {'role': 'system', 'content': 'You extract strict JSON only.'},
            {'role': 'user', 'content': prompt},
        ],
    )
    content = completion.choices[0].message.content or '{}'
    return json.loads(content)


def _extract_mock(html_text: str) -> Dict[str, Any]:
    """Extract the minimum viable listing payload using BeautifulSoup and regexes."""
    soup = BeautifulSoup(html_text, 'html.parser')

    title = (soup.find('h1') or soup.find('title'))
    title_text = title.get_text(' ', strip=True) if title else None

    description_candidates = [
        soup.select_one('.description'),
        soup.find('meta', attrs={'name': 'description'}),
        soup.find('p'),
    ]
    description = None
    for candidate in description_candidates:
        if candidate is None:
            continue
        if getattr(candidate, 'name', '') == 'meta':
            description = candidate.get('content')
        else:
            description = candidate.get_text(' ', strip=True)
        if description:
            break

    full_text = soup.get_text(' ', strip=True)
    location_match = re.search(r'([A-Za-z ]+),\s*([A-Za-z ]+)', full_text)
    governorate = location_match.group(1).strip() if location_match else None
    city = location_match.group(2).strip() if location_match else None

    surface_match = re.search(r'(\d{2,4})\s?(m2|m\^2|sqm)', full_text, re.IGNORECASE)
    rooms_match = re.search(r'(\d+)\s?(rooms|room|pieces)', full_text, re.IGNORECASE)
    baths_match = re.search(r'(\d+)\s?(bath|bathroom)', full_text, re.IGNORECASE)

    ptype = None
    for token in ('Appartement', 'Maison', 'Terrain', 'Commercial'):
        if token.lower() in full_text.lower():
            ptype = token
            break

    condition = None
    for cond in ('New', 'Excellent', 'Good', 'Fair', 'Needs Renovation'):
        if cond.lower() in full_text.lower():
            condition = cond
            break

    return {
        'title': title_text,
        'description': description,
        'property_type': ptype,
        'governorate': governorate,
        'city': city,
        'neighborhood': None,
        'surface_m2': float(surface_match.group(1)) if surface_match else None,
        'bedrooms': int(rooms_match.group(1)) - 1 if rooms_match else 2,
        'bathrooms': int(baths_match.group(1)) if baths_match else 1,
        'condition': condition or 'Good',
        'transaction_type': 'sale',
        'amenities': {
            'has_pool': 'pool' in full_text.lower(),
            'has_garden': 'garden' in full_text.lower(),
            'has_parking': 'parking' in full_text.lower(),
            'sea_view': 'sea view' in full_text.lower() or 'vue mer' in full_text.lower(),
            'elevator': 'elevator' in full_text.lower() or 'ascenseur' in full_text.lower(),
        },
        'posted_at': None,
    }


def extraction_node(state: ListingAgentState) -> ListingAgentState:
    """Choose the best extraction strategy and store the parsed listing data."""
    _append_log(state, 'extraction_node: start')
    html_text = state.get('raw_html', '')
    if not html_text:
        raise ValueError('raw_html is missing')

    use_openai = bool(os.getenv('OPENAI_API_KEY'))
    try:
        if use_openai:
            extracted = with_retries(_extract_with_openai, html_text, retries=2)
            _append_log(state, 'extraction_node: used OpenAI extraction')
        else:
            extracted = _extract_mock(html_text)
            _append_log(state, 'extraction_node: used mock extraction')
    except Exception as exc:
        _append_log(state, f'extraction_node: extraction failed, fallback to mock. error={exc}')
        extracted = _extract_mock(html_text)

    extracted.pop('price', None)
    state['extracted_data'] = extracted
    return state

In [ ]:
def structuring_node(state: ListingAgentState) -> ListingAgentState:
    """Normalize extracted fields into the internal listing schema."""
    _append_log(state, 'structuring_node: start')
    data = dict(state.get('extracted_data') or {})

    normalized = normalize_tunisian_data({
        'property_type': data.get('property_type'),
        'description': data.get('description'),
        'location': f"{data.get('governorate') or ''}, {data.get('city') or ''}",
        'surface': data.get('surface_m2'),
    })

    structured = {
        # Align with dataset flavor from final_listings_wrangled.csv
        'record_id': None,
        'source': 'LangGraphAgent',
        'source_file': 'runtime',
        'listing_url': state.get('source_url'),
        'title': data.get('title'),
        'description': data.get('description'),
        'transaction_type': (data.get('transaction_type') or 'sale').lower(),
        'property_type': normalized.get('property_type') or data.get('property_type') or 'Appartement',
        'price_tnd': None,
        'surface_m2': data.get('surface_m2') or normalized.get('surface_area'),
        'price_per_m2': None,
        'rooms': data.get('bedrooms', 0) + (0 if str(data.get('property_type', '')).lower() == 'terrain' else 1),
        'bedrooms': data.get('bedrooms'),
        'bathrooms': data.get('bathrooms'),
        'governorate': normalized.get('governorate') or data.get('governorate'),
        'city': data.get('city'),
        'neighborhood': data.get('neighborhood'),
        'location_raw': None,
        'posted_at': data.get('posted_at'),
        'scraped_at': _utc_now_iso(),
        'currency': 'TND',
        'image_url': None,
        'image_count': 0,
        'amenities': data.get('amenities', {}),
    }

    state['structured_data'] = structured
    return state


def validation_node(state: ListingAgentState) -> ListingAgentState:
    """Fill missing required fields and enforce basic value constraints."""
    _append_log(state, 'validation_node: start')
    listing = dict(state.get('structured_data') or {})
    warnings: List[str] = []

    if not listing.get('title'):
        listing['title'] = 'Untitled listing'
        warnings.append('missing_title_defaulted')

    if not listing.get('description'):
        listing['description'] = ''
        warnings.append('missing_description_defaulted')

    if not listing.get('governorate'):
        listing['governorate'] = 'Tunis'
        warnings.append('missing_governorate_defaulted')

    if not listing.get('city'):
        listing['city'] = listing['governorate']
        warnings.append('missing_city_defaulted')

    surface = listing.get('surface_m2')
    if surface is None:
        listing['surface_m2'] = 120.0
        warnings.append('missing_surface_defaulted')
    else:
        listing['surface_m2'] = float(max(10.0, min(float(surface), 2000.0)))

    bedrooms = listing.get('bedrooms')
    bathrooms = listing.get('bathrooms')
    listing['bedrooms'] = int(max(0, int(bedrooms if bedrooms is not None else 2)))
    listing['bathrooms'] = int(max(0, int(bathrooms if bathrooms is not None else 1)))

    condition_allowed = {'New', 'Excellent', 'Good', 'Fair', 'Needs Renovation'}
    if listing.get('condition') not in condition_allowed:
        listing['condition'] = 'Good'
        warnings.append('condition_normalized_to_good')

    ptype = str(listing.get('property_type') or '').title()
    if ptype not in {'Terrain', 'Maison', 'Appartement', 'Commercial'}:
        listing['property_type'] = 'Appartement'
        warnings.append('property_type_defaulted_appartement')
    else:
        listing['property_type'] = ptype

    listing['warnings'] = warnings
    state['warnings'] = warnings
    state['validated_data'] = listing
    _append_log(state, f'validation_node: completed with {len(warnings)} warning(s)')
    return state


def feature_engineering_node(state: ListingAgentState) -> ListingAgentState:
    """Derive heuristic features used by the valuation step."""
    _append_log(state, 'feature_engineering_node: start')
    listing = dict(state.get('validated_data') or {})
    desc = str(listing.get('description') or '').lower()

    posted_at = listing.get('posted_at')
    property_age = None
    if posted_at:
        try:
            posted_dt = datetime.fromisoformat(str(posted_at).replace('Z', '+00:00'))
            property_age = max(0, (datetime.now(timezone.utc) - posted_dt).days)
        except Exception:
            property_age = None

    has_sea_view = bool(listing.get('amenities', {}).get('sea_view')) or ('sea view' in desc) or ('vue mer' in desc)

    location_score_map = {
        'tunis': 0.95,
        'nabeul': 0.88,
        'sousse': 0.86,
        'sfax': 0.82,
        'ariana': 0.84,
    }
    governorate = str(listing.get('governorate') or '').lower()
    location_score = location_score_map.get(governorate, 0.75)

    keyword_features = {
        'kw_luxury': int(any(w in desc for w in ['luxury', 'high-end', 'premium', 'de luxe'])),
        'kw_renovated': int(any(w in desc for w in ['renovated', 'refait', 'renove'])),
    }

    amenities = dict(listing.get('amenities') or {})
    enriched = {
        **listing,
        'property_age': property_age,
        'has_sea_view': has_sea_view,
        'location_score': location_score,
        **keyword_features,
        'amenities': amenities,
    }

    state['enriched_features'] = enriched
    return state

In [16]:
def _mock_price_from_features(features: Dict[str, Any]) -> Dict[str, Any]:
    """Produce a fallback valuation when the model service is unavailable."""
    surface = float(features.get('surface_m2') or 100.0)
    loc = float(features.get('location_score') or 0.75)
    condition = str(features.get('condition') or 'Good')
    cond_factor = {
        'Needs Renovation': 0.8,
        'Fair': 0.9,
        'Good': 1.0,
        'Excellent': 1.12,
        'New': 1.18,
    }.get(condition, 1.0)

    base_m2 = 1500.0 * loc
    amenity_bonus = 1.0
    amenity_bonus += 0.06 if bool(features.get('has_sea_view')) else 0.0
    amenity_bonus += 0.03 if bool(features.get('amenities', {}).get('has_parking')) else 0.0
    amenity_bonus += 0.04 if bool(features.get('amenities', {}).get('has_pool')) else 0.0
    amenity_bonus += 0.02 * int(features.get('kw_luxury', 0))
    amenity_bonus += 0.02 * int(features.get('kw_renovated', 0))

    estimated_price = max(1.0, surface * base_m2 * cond_factor * amenity_bonus)
    confidence = max(0.45, min(0.9, 0.6 + (loc - 0.7) * 0.5))

    return {
        'estimated_price': float(round(estimated_price, 2)),
        'price_per_m2': float(round(estimated_price / max(surface, 1.0), 2)),
        'confidence': float(round(confidence, 3)),
        'mode': 'mock_regression',
        'warnings': ['mock_model_used'],
        'model_info': {'name': 'heuristic_mock_model'},
    }


def valuation_node(state: ListingAgentState) -> ListingAgentState:
    """Call the production valuation service and fall back to the heuristic model."""
    _append_log(state, 'valuation_node: start')
    features = dict(state.get('enriched_features') or {})

    payload = ValuationPayload(
        property_type=str(features.get('property_type') or 'Appartement'),
        transaction_type=str(features.get('transaction_type') or 'sale'),
        governorate=str(features.get('governorate') or 'Tunis'),
        city=str(features.get('city') or 'Tunis'),
        neighborhood=str(features.get('neighborhood') or ''),
        size_m2=float(features.get('surface_m2') or 120.0),
        bedrooms=int(features.get('bedrooms') or 2),
        bathrooms=int(features.get('bathrooms') or 1),
        condition=str(features.get('condition') or 'Good'),
        has_pool=bool(features.get('amenities', {}).get('has_pool')),
        has_garden=bool(features.get('amenities', {}).get('has_garden')),
        has_parking=bool(features.get('amenities', {}).get('has_parking')),
        sea_view=bool(features.get('has_sea_view')),
        elevator=bool(features.get('amenities', {}).get('elevator')),
        description=str(features.get('description') or ''),
        uploaded_images_count=int(features.get('image_count') or 0),
        image_refs=[],
    )

    try:
        if VALUATION_SERVICE is None:
            raise RuntimeError('ValuationService unavailable')
        valuation_result = with_retries(
            VALUATION_SERVICE.estimate,
            payload,
            external_warnings=state.get('warnings', []),
            retries=2,
        )
        predicted = {
            'estimated_price': float(valuation_result.get('estimated_price', 0.0)),
            'price_per_m2': float(valuation_result.get('price_per_m2', 0.0)),
            'confidence': float(valuation_result.get('confidence_score', 0.6)),
            'mode': valuation_result.get('prediction_mode', 'estate_service'),
            'warnings': valuation_result.get('warnings', []),
            'model_info': valuation_result.get('model_info', {}),
            'raw_response': valuation_result,
        }
        _append_log(state, 'valuation_node: used EstateMind ValuationService')
    except Exception as exc:
        _append_log(state, f'valuation_node: fallback to mock model. error={exc}')
        predicted = _mock_price_from_features(features)

    state['predicted_price'] = predicted
    return state


def postprocessing_node(state: ListingAgentState) -> ListingAgentState:
    """Combine validated data and valuation output into a presentation-ready record."""
    _append_log(state, 'postprocessing_node: start')
    validated = dict(state.get('validated_data') or {})
    enriched = dict(state.get('enriched_features') or {})
    predicted = dict(state.get('predicted_price') or {})

    estimated_price = float(predicted.get('estimated_price') or 0.0)
    confidence = float(predicted.get('confidence') or 0.6)

    margin = max(0.07, 0.22 - confidence * 0.12)
    low = round(estimated_price * (1 - margin), 2)
    high = round(estimated_price * (1 + margin), 2)

    explanation = (
        f"Predicted from location ({enriched.get('governorate', 'N/A')}), "
        f"surface ({enriched.get('surface_m2', 'N/A')} m2), condition ({enriched.get('condition', 'N/A')}), "
        f"and amenities (sea_view={enriched.get('has_sea_view', False)})."
    )

    final_listing = {
        **validated,
        'predicted_price_tnd': estimated_price,
        'predicted_price_range_tnd': {'low': low, 'high': high},
        'confidence_score': confidence,
        'price_per_m2_predicted': predicted.get('price_per_m2'),
        'explanation_short': explanation,
        'prediction_mode': predicted.get('mode'),
        'prediction_warnings': sorted(set((state.get('warnings') or []) + (predicted.get('warnings') or []))),
        'model_info': predicted.get('model_info', {}),
    }

    state['final_listing'] = final_listing
    return state


def storage_node(state: ListingAgentState) -> ListingAgentState:
    """Persist the final listing to memory and the local SQLite store."""
    _append_log(state, 'storage_node: start')
    listing = dict(state.get('final_listing') or {})

    listing_id = str(uuid.uuid4())
    created_at = _utc_now_iso()

    listing['id'] = listing_id
    listing['created_at'] = created_at

    HISTORY_STORE.append({'id': listing_id, 'created_at': created_at, 'listing': listing})

    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        """
        INSERT INTO listings (
            id, created_at, source_url, title, property_type, governorate, city, neighborhood,
            surface_m2, bedrooms, bathrooms, condition, predicted_price_tnd, confidence, payload_json
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            listing_id,
            created_at,
            str(state.get('source_url') or ''),
            str(listing.get('title') or ''),
            str(listing.get('property_type') or ''),
            str(listing.get('governorate') or ''),
            str(listing.get('city') or ''),
            str(listing.get('neighborhood') or ''),
            float(listing.get('surface_m2') or 0.0),
            int(listing.get('bedrooms') or 0),
            int(listing.get('bathrooms') or 0),
            str(listing.get('condition') or ''),
            float(listing.get('predicted_price_tnd') or 0.0),
            float(listing.get('confidence_score') or 0.0),
            json.dumps(listing, ensure_ascii=False),
        ),
    )
    conn.commit()
    conn.close()

    state['final_listing'] = listing
    _append_log(state, f'storage_node: stored listing id={listing_id}')
    return state

In [17]:
def build_graph() -> Any:
    """Construct and compile the LangGraph pipeline used by the notebook."""
    graph = StateGraph(ListingAgentState)

    graph.add_node('scraper', scraper_node)
    graph.add_node('extraction', extraction_node)
    graph.add_node('structuring', structuring_node)
    graph.add_node('validation', validation_node)
    graph.add_node('feature_engineering', feature_engineering_node)
    graph.add_node('valuation', valuation_node)
    graph.add_node('postprocessing', postprocessing_node)
    graph.add_node('storage', storage_node)

    graph.set_entry_point('scraper')
    graph.add_edge('scraper', 'extraction')
    graph.add_edge('extraction', 'structuring')
    graph.add_edge('structuring', 'validation')
    graph.add_edge('validation', 'feature_engineering')
    graph.add_edge('feature_engineering', 'valuation')
    graph.add_edge('valuation', 'postprocessing')
    graph.add_edge('postprocessing', 'storage')
    graph.add_edge('storage', END)

    return graph.compile()


listing_graph = build_graph()

In [ ]:
# Example single run from real scraper sources in src/scripts
sample_urls = [
    'https://www.tayara.tn/c/immobilier',
    'https://www.mubawab.tn/fr/listing-promotion:p:1',
    'http://www.tunisie-annonce.com/AnnoncesImmobilier.asp',
    'https://www.tecnocasa.tn/vendre/immeubles/nord-est-ne/grand-tunis.html',
]

sample_url = sample_urls[0]
initial_state: ListingAgentState = {
    'source_url': sample_url,
    'logs': [],
}

result_state = listing_graph.invoke(initial_state)
result_state['final_listing']

INFO:langgraph_real_estate_agent:scraper_node: start
INFO:langgraph_real_estate_agent:scraper_node: fetched 498715 chars from https://www.tayara.tn/c/immobilier
INFO:langgraph_real_estate_agent:extraction_node: start
INFO:langgraph_real_estate_agent:extraction_node: used mock extraction
INFO:langgraph_real_estate_agent:structuring_node: start
INFO:langgraph_real_estate_agent:validation_node: start
INFO:langgraph_real_estate_agent:validation_node: completed with 2 warning(s)
INFO:langgraph_real_estate_agent:feature_engineering_node: start
INFO:langgraph_real_estate_agent:valuation_node: start
INFO:langgraph_real_estate_agent:valuation_node: used EstateMind ValuationService
INFO:langgraph_real_estate_agent:postprocessing_node: start
INFO:langgraph_real_estate_agent:storage_node: start
INFO:langgraph_real_estate_agent:storage_node: stored listing id=cfb37b04-324e-4d3e-9f05-0c30904a1edf


{'record_id': None,
 'source': 'LangGraphAgent',
 'source_file': 'runtime',
 'listing_url': 'https://www.tayara.tn/c/immobilier',
 'title': 'Appartements',
 'description': "Tayara, achat et vente gratuitement des voitures, de l'immobilier, des smartphones, des ordinateurs, des électroménagers et plus.",
 'transaction_type': 'sale',
 'property_type': 'Appartement',
 'price_tnd': None,
 'surface_m2': 120.0,
 'price_per_m2': None,
 'rooms': 3,
 'bedrooms': 2,
 'bathrooms': 1,
 'governorate': 'Khreidine, La Goulette Appartements Tunis',
 'city': 'La Goulette Appartements Tunis',
 'neighborhood': None,
 'location_raw': None,
 'posted_at': None,
 'scraped_at': '2026-04-09T14:44:56.897116+00:00',
 'currency': 'TND',
 'image_url': None,
 'image_count': 0,
 'amenities': {'has_pool': False,
  'has_garden': False,
  'has_parking': True,
  'sea_view': False,
  'elevator': True},
 'condition': 'Good',
 'warnings': ['missing_surface_defaulted', 'condition_normalized_to_good'],
 'predicted_price_tnd'

In [19]:
# Inspect pipeline logs
for entry in result_state.get('logs', []):
    print(entry)

2026-04-09T14:44:55.411842+00:00 | scraper_node: start
2026-04-09T14:44:56.798537+00:00 | scraper_node: fetched 498715 chars from https://www.tayara.tn/c/immobilier
2026-04-09T14:44:56.800343+00:00 | extraction_node: start
2026-04-09T14:44:56.894783+00:00 | extraction_node: used mock extraction
2026-04-09T14:44:56.896302+00:00 | structuring_node: start
2026-04-09T14:44:56.897536+00:00 | validation_node: start
2026-04-09T14:44:56.898098+00:00 | validation_node: completed with 2 warning(s)
2026-04-09T14:44:56.899006+00:00 | feature_engineering_node: start
2026-04-09T14:44:56.900555+00:00 | valuation_node: start
2026-04-09T14:44:57.650032+00:00 | valuation_node: used EstateMind ValuationService
2026-04-09T14:44:57.651972+00:00 | postprocessing_node: start
2026-04-09T14:44:57.652924+00:00 | storage_node: start
2026-04-09T14:44:57.664843+00:00 | storage_node: stored listing id=cfb37b04-324e-4d3e-9f05-0c30904a1edf


In [ ]:
# Bonus: async batch processing for multiple listing URLs
async def process_listing_url_async(url: str) -> ListingAgentState:
    """Run the full graph for a single listing URL using the async API."""
    state: ListingAgentState = {'source_url': url, 'logs': []}
    return await listing_graph.ainvoke(state)


async def batch_process_async(urls: List[str], concurrency: int = 5) -> List[ListingAgentState]:
    """Process multiple URLs concurrently while capping the number of in-flight tasks."""
    semaphore = asyncio.Semaphore(concurrency)

    async def _bounded(url: str) -> ListingAgentState:
        async with semaphore:
            return await process_listing_url_async(url)

    tasks = [_bounded(url) for url in urls]
    return await asyncio.gather(*tasks, return_exceptions=False)